# Qwen3-8B LoRA — IFBench Instruction Following
**Project**: The Efficiency Threshold — CS224N

This notebook evaluates the **context-weight tradeoff** for instruction following:
- **Task**: Precise instruction following (output constraint satisfaction)
- **Model**: `Qwen/Qwen3-8B` (base, not instruct — isolates LoRA adaptation)
- **Method**: LoRA fine-tuning (primary comparator for IFBench per proposal)
- **Training data**: `allenai/IF_multi_constraints_upto5` — official IFBench training set (95k examples)
- **Eval data**: `allenai/IFBench_test` — 300 held-out OOD constraint examples (never trained on)
- **Splits**: N ∈ {16, 32, 64, 128, 256}, seeds {0, 1, 2} — deterministic, nested, local only
- **Metric**: Prompt-level loose accuracy (fraction of prompts where ALL constraints are satisfied)

### Key difference from classification notebooks (RTE, Financial PhraseBank)
IFBench is a **generation** benchmark — there is no discrete gold label. Instead:
1. Training requires (prompt, constraint-satisfying response) pairs. This notebook **auto-generates and filters gold responses** from the training pool using the IFBench verifier before fine-tuning.
2. Evaluation uses the AllenAI constraint verification functions, not exact-match accuracy.

In [1]:
!nvidia-smi

Tue Feb 24 01:34:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3. Load Datasets

- **Training pool**: `allenai/IF_multi_constraints_upto5` — 95k constraint-augmented instruction prompts.
  - Columns: `key`, `messages` (user prompt in chat format), `ground_truth` (instruction_id + kwargs for verification), `constraint_type`, `constraint`
- **Eval set**: `allenai/IFBench_test` — 300 OOD constraint prompts, held out entirely.
  - Columns: `key`, `prompt`, `instruction_id_list`, `kwargs`

In [3]:
from datasets import load_dataset
import json

# ---- Training pool ----
train_pool_ds = load_dataset("allenai/IF_multi_constraints_upto5", split="train")
print(f"Training pool size : {len(train_pool_ds)}")
print(f"Columns            : {train_pool_ds.column_names}")
print()

# ---- Eval set ----
eval_ds = load_dataset("allenai/IFBench_test", split="train")  # only split is 'train'
print(f"Eval set size      : {len(eval_ds)}")
print(f"Columns            : {eval_ds.column_names}")
print()
print("--- Training pool example ---")
ex = train_pool_ds[0]
print("prompt  :", ex["messages"][0]["content"][:200], "...")
print("gt      :", ex["ground_truth"])
print()
print("--- Eval example ---")
ex2 = eval_ds[0]
print("prompt  :", ex2["prompt"][:200], "...")
print("inst_ids:", ex2["instruction_id_list"])

Training pool size : 95373
Columns            : ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint']

Eval set size      : 300
Columns            : ['key', 'prompt', 'instruction_id_list', 'kwargs']

--- Training pool example ---
prompt  : Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be connected using hyphens, with no spaces between them. The last word of your response should be t ...
gt      : [{'instruction_id': ['detectable_format:sentence_hyphens', 'last_word:last_word_answer'], 'kwargs': [None, {'last_word': 'brief'}]}]

--- Eval example ---
prompt  : What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally  ...
inst_ids: ['count:keywords_multiple']


## 4. IFBench Evaluation Utilities

Import the constraint verification library from the cloned IFBench repo.
- `test_instruction_following_loose`: checks whether a response satisfies all constraints; returns per-instruction True/False
- **Prompt-level loose accuracy**: fraction of prompts where ALL constraints pass (consistent with the IFBench paper's primary metric)

In [4]:
!git clone https://github.com/allenai/IFBench.git ifbench_repo 2>&1 || echo "Already exists"

fatal: destination path 'ifbench_repo' already exists and is not an empty directory.
Already exists


In [5]:
import sys, os

# Verify the clone worked and find evaluation_lib.py
ifbench_path = os.path.abspath("ifbench_repo")

print("IFBench repo path:", ifbench_path)
print("Files in repo:", os.listdir(ifbench_path))

# Add to path using the absolute path — safe across working directory changes
if ifbench_path not in sys.path:
    sys.path.insert(0, ifbench_path)

from evaluation_lib import test_instruction_following_loose
print("✓ evaluation_lib imported successfully")

IFBench repo path: /content/ifbench_repo
Files in repo: ['run_eval.py', 'evaluation_lib.py', 'LICENSE', 'eval', '__pycache__', '.git', '.gitignore', '.nltk_data', 'Precise_IF_Generalization_Abilities.pdf', '.env.example', 'uv.lock', 'requirements.txt', 'instructions_registry.py', 'pyproject.toml', 'data', 'generate_responses.py', 'instructions.py', 'instructions_test.py', 'README.md', '.coverage', 'config.py', 'instructions_util.py']
✓ evaluation_lib imported successfully


In [3]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.3 MB/s eta 0:00:00


In [4]:
!pip install syllapy

In [6]:
import sys
sys.path.insert(0, "ifbench_repo")

from evaluation_lib import test_instruction_following_loose

def check_constraint_satisfaction(prompt: str, instruction_id_list: list, kwargs: list, response: str) -> bool:
    """
    Returns True if the response satisfies ALL constraints in a prompt.
    Uses IFBench's loose evaluation (equivalent to the paper's primary metric).
    """
    # Build the input dict expected by the IFBench evaluation library
    inp = {
        "prompt": prompt,
        "instruction_id_list": instruction_id_list,
        "kwargs": kwargs,
    }
    result = test_instruction_following_loose(inp, response)
    return all(result.follow_instruction_list)


def eval_ifbench(model, tokenizer, eval_examples, max_new_tokens=512):
    """
    Evaluate a model on IFBench_test.
    Returns prompt-level loose accuracy and per-example results.
    """
    import torch, time

    model.eval()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    results = []
    total_gen_time = 0.0
    total_new_tokens = 0

    for ex in eval_examples:
        prompt_text = ex["prompt"]
        # Format as a simple user message
        formatted = f"{prompt_text}"

        inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
        prompt_len = int(inputs["input_ids"].shape[-1])

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        gen_ids = out[0][prompt_len:]
        response = tokenizer.decode(gen_ids, skip_special_tokens=True)
        new_toks = int(gen_ids.shape[-1])
        gen_time = t1 - t0

        satisfied = check_constraint_satisfaction(
            prompt_text,
            ex["instruction_id_list"],
            ex["kwargs"],
            response,
        )

        results.append({
            "key": ex.get("key", ""),
            "satisfied": satisfied,
            "response": response,
            "prompt_tokens": prompt_len,
            "new_tokens": new_toks,
            "gen_time_sec": gen_time,
        })
        total_gen_time += gen_time
        total_new_tokens += new_toks

    n = len(results)
    accuracy = sum(r["satisfied"] for r in results) / n
    avg_gen_time = total_gen_time / n
    avg_prompt_tokens = sum(r["prompt_tokens"] for r in results) / n

    metrics = {
        "accuracy": accuracy,
        "eval_size": n,
        "gen_tokens_per_sec": total_new_tokens / total_gen_time if total_gen_time > 0 else 0.0,
        "avg_gen_time_sec": avg_gen_time,
        "total_gen_time_sec": total_gen_time,
        "avg_prompt_tokens": avg_prompt_tokens,
        "peak_vram_mb": torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0,
    }
    return metrics, results

print("IFBench evaluation utilities loaded.")

IFBench evaluation utilities loaded.


In [7]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install bitsandbytes accelerate

In [15]:
!pip install flash-attn

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=256026557 sha256=51100460844b0cb4bd5adc38af60c56adff8904508861aeb5ef7a1c2e41f529a
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# The CUDA cache is cleared to prevent leftover memoryallocations from causing failures
torch.cuda.empty_cache()
# A robust 4-bit quantization configuration is defined to minimize the memory footprint
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
# The tokenizer is initialized
model_id = "Qwen/Qwen3-8B"
gen_tokenizer = AutoTokenizer.from_pretrained(model_id,use_fast=True)
# The model is loaded with quantization, lower precision, and automatic device mapping applied
gen_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
print("The model has been successfully loaded without exceeding memory limits.")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The model has been successfully loaded without exceeding memory limits.


## 5. Generate Gold Responses for Training

**Why this step exists**: IFBench is a generation benchmark — there are no pre-written gold responses.
For LoRA SFT we need (prompt, good_response) pairs.

Full version

In [13]:
# @title
import os, json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import ast # Import the ast module

# --- Fix for check_constraint_satisfaction ---
# The previous definition was incorrect for the IFBench library signature.
# Redefining it here locally to fix the execution.

class SimpleInputExample:
    def __init__(self, prompt, instruction_id_list, kwargs):
        self.prompt = prompt
        self.instruction_id_list = instruction_id_list
        self.kwargs = kwargs

def check_constraint_satisfaction(prompt: str, instruction_id_list: list, kwargs: list, response: str) -> bool:
    """
    Returns True if the response satisfies ALL constraints in a prompt.
    Uses IFBench's loose evaluation (equivalent to the paper's primary metric).
    """
    # Wrap input in an object with attributes as expected by evaluation_lib
    inp = SimpleInputExample(
        prompt=prompt,
        instruction_id_list=instruction_id_list,
        kwargs=kwargs
    )
    # The evaluator expects a dictionary mapping prompt -> response
    prompt_to_response = {prompt: response}

    # Robust import to avoid UnboundLocalError due to shadowing
    import sys
    if "ifbench_repo" not in sys.path:
        sys.path.insert(0, os.path.abspath("ifbench_repo"))

    # Import the module directly to avoid local/global scope collision
    import evaluation_lib

    result = evaluation_lib.test_instruction_following_loose(inp, prompt_to_response)
    return all(result.follow_instruction_list)
# ---------------------------------------------

GOLD_RESPONSES_PATH = Path("data/ifbench_gold_responses.jsonl")
GOLD_RESPONSES_PATH.parent.mkdir(parents=True, exist_ok=True)

REGEN = not GOLD_RESPONSES_PATH.exists()  # Set to True to force regeneration
MAX_CANDIDATES = 100  # Sample from the 95k pool to keep runtime manageable
MAX_GOLD_NEEDED = 16  # We only need enough to cover N=256 x 3 seeds with headroom

if not REGEN:
    print(f"Loading existing gold responses from {GOLD_RESPONSES_PATH}")
    with GOLD_RESPONSES_PATH.open() as f:
        gold_examples = [json.loads(l) for l in f]
    print(f"Loaded {len(gold_examples)} gold (prompt, response) pairs.")
else:
    print("Generating gold responses with Qwen3-8B...")

    #bnb_config = BitsAndBytesConfig(
    #    load_in_4bit=True,
    #    bnb_4bit_use_double_quant=True,
    #    bnb_4bit_quant_type="nf4",
    #    bnb_4bit_compute_dtype=torch.bfloat16
    #)

    #gen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B", use_fast=True)
    #gen_model = AutoModelForCausalLM.from_pretrained(
    #    "Qwen/Qwen3-4B",
    #    quantization_config=bnb_config,
    #    torch_dtype=torch.bfloat16,
    #    device_map="auto",
    #)
    gen_model.eval()
    if gen_tokenizer.pad_token is None:
        gen_tokenizer.pad_token = gen_tokenizer.eos_token

    # Sample candidate prompts from the training pool
    import random
    rng = random.Random(42)
    indices = list(range(len(train_pool_ds)))
    rng.shuffle(indices)
    candidates = [train_pool_ds[i] for i in indices[:MAX_CANDIDATES]]

    gold_examples = []
    for idx, ex in enumerate(candidates):
        if len(gold_examples) >= MAX_GOLD_NEEDED:
            break

        prompt_text = ex["messages"][0]["content"]
        # Parse the ground_truth string into a Python object
        # The dataset stores ground_truth as a string with single quotes, which is not valid JSON.
        # Use ast.literal_eval to safely parse it as a Python literal.
        gt = ast.literal_eval(ex["ground_truth"])[0]  # {instruction_id: [...], kwargs: [...]}

        inputs = gen_tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=gen_tokenizer.pad_token_id,
                eos_token_id=gen_tokenizer.eos_token_id,
            )
        prompt_len = inputs["input_ids"].shape[-1]
        response = gen_tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

        # Only keep constraint-satisfying responses
        try:
            satisfied = check_constraint_satisfaction(
                prompt_text,
                gt["instruction_id"],
                gt["kwargs"],
                response,
            )
        except Exception as e:
            # KeyErrors for missing instructions or other eval lib issues
            # print(f"Skipping candidate {ex['key']} due to eval error: {e}")
            satisfied = False

        if satisfied:
            gold_examples.append({
                "key": ex["key"],
                "prompt": prompt_text,
                "response": response,
                "instruction_id": gt["instruction_id"],
                "kwargs": gt["kwargs"],
            })

        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx+1}/{MAX_CANDIDATES} candidates, {len(gold_examples)} gold so far")

    # Save
    with GOLD_RESPONSES_PATH.open("w") as f:
        for g in gold_examples:
            f.write(json.dumps(g) + "\n")

    # Free VRAM
    del gen_model
    torch.cuda.empty_cache()

    print(f"\nSaved {len(gold_examples)} gold examples to {GOLD_RESPONSES_PATH}")

assert len(gold_examples) >= 8, (
    f"Only {len(gold_examples)} gold examples found — need at least 256 for N=256 splits. "
    "Increase MAX_CANDIDATES or lower MAX_GOLD_NEEDED."
)
print(f"Gold pool size: {len(gold_examples)} — sufficient for all N splits.")

Generating gold responses with Qwen3-8B...
  Processed 100/100 candidates, 0 gold so far

Saved 0 gold examples to data/ifbench_gold_responses.jsonl


AssertionError: Only 0 gold examples found — need at least 256 for N=256 splits. Increase MAX_CANDIDATES or lower MAX_GOLD_NEEDED.

## 6. Generate Deterministic Splits

Inline split generation:
- Unstratified (no discrete labels)
- Nested by prefix property: N=16 ⊆ N=32 ⊆ ... ⊆ N=256
- Splits are over the **gold training examples**, not the eval set

In [ ]:
import random, os
from pathlib import Path

Ns    = [16, 32, 64, 128, 256]
SEEDS = [0, 1, 2]
SPLITS_DIR = Path("data/splits_out/ifbench")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

n_total = len(gold_examples)
assert max(Ns) <= n_total, f"Not enough gold examples ({n_total}) for N={max(Ns)}"

all_splits = {}  # {seed: {N: [indices]}}

for seed in SEEDS:
    rng = random.Random(seed)
    shuffled = list(range(n_total))
    rng.shuffle(shuffled)

    splits_for_seed = {}
    for N in sorted(Ns):
        # sorted prefix for nesting guarantee
        splits_for_seed[N] = sorted(shuffled[:N])

    # Write JSONL files
    for N in Ns:
        out_path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
        with out_path.open("w") as f:
            for i in splits_for_seed[N]:
                f.write(json.dumps(gold_examples[i]) + "\n")

    all_splits[seed] = splits_for_seed

# Nesting check
print("Nesting check:")
for seed in SEEDS:
    for a, b in zip(Ns[:-1], Ns[1:]):
        A = set(all_splits[seed][a])
        B = set(all_splits[seed][b])
        assert A.issubset(B), f"Nesting FAILED: seed={seed} N={a} not subset of N={b}"
        print(f"  seed={seed}  N={a:>3} ⊆ N={b:>3}: ✓")

print(f"\nAll splits written to {SPLITS_DIR}")
print("Files:", sorted([p.name for p in SPLITS_DIR.glob("*.jsonl")]))

## 7. Load Qwen3-8B Base Model

Using thebase to isolate adaptation through LoRA weight updates rather than pre-existing instruction-following priors, consistent with the project proposal.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded:", BASE_MODEL)
print("Device:", next(model.parameters()).device)
print("Params:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## 8. Zero-Shot Baseline

Evaluate the unmodified Qwen3-8B base on IFBench_test to establish the lower bound before any adaptation.

In [ ]:
eval_examples = [eval_ds[i] for i in range(len(eval_ds))]

zero_shot_metrics, _ = eval_ifbench(model, tokenizer, eval_examples)

zero_shot_result = {
    "method": "zero_shot",
    "model": BASE_MODEL,
    "N": 0,
    "seed": -1,
    **zero_shot_metrics,
    "train_time_sec": 0.0,
}
print("Zero-shot result:")
print(json.dumps(zero_shot_result, indent=2))

## 9. LoRA Training & Evaluation Loop

For each N in {16, 32, 64, 128, 256}:
1. Load the N-example split from disk
2. Fine-tune Qwen3-8B with LoRA adapters on the (prompt, response) pairs
3. Evaluate on all 300 IFBench_test examples
4. Record accuracy, latency, VRAM

**Training format**: The model sees `<prompt>` and is trained to produce `<response>`. Token loss is computed only on the response tokens (the prompt is masked out as context).

In [ ]:
import time
from pathlib import Path
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType


def read_jsonl(path: Path):
    with path.open("r") as f:
        return [json.loads(l) for l in f if l.strip()]


def make_train_text(ex: dict) -> str:
    """
    Format a gold (prompt, response) pair as a single training string.
    The model learns to continue the prompt with the satisfying response.
    """
    return f"{ex['prompt']}\n\n{ex['response']}"


def build_train_dataset(N: int, seed: int = 0) -> Dataset:
    path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
    rows = read_jsonl(path)
    return Dataset.from_dict({"text": [make_train_text(r) for r in rows]})


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
    )


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


def make_lora_model(base_model):
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    m = get_peft_model(base_model, lora_cfg)
    m.print_trainable_parameters()
    return m


def train_lora_for_N(N: int, seed: int = 0, out_dir: str = None):
    train_ds = build_train_dataset(N, seed=seed).map(
        tokenize_batch, batched=True, remove_columns=["text"]
    )

    # Reload fresh base model for each run to avoid adapter accumulation
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lora_model = make_lora_model(base)
    lora_model.train()

    if out_dir is None:
        out_dir = f"artifacts/lora_ifbench_N{N}_seed{seed}"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        # Fixed optimization budget across N (same as RTE notebook)
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="no",
        bf16=True,
        report_to=[],
        seed=seed,
    )

    trainer = Trainer(
        model=lora_model,
        args=args,
        train_dataset=train_ds,
        data_collator=data_collator,
    )

    t0 = time.perf_counter()
    trainer.train()
    t1 = time.perf_counter()

    lora_model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    return lora_model, (t1 - t0), out_dir


print("Training utilities loaded.")

In [ ]:
# -------------------------------------------------------
# Run the full N-sweep for seed=0
# (Run cells below for seeds 1 and 2 to build full results)
# -------------------------------------------------------
lora_results = [zero_shot_result]  # include zero-shot baseline

for N in [16, 32, 64, 128, 256]:
    print(f"\n{'='*60}")
    print(f"Training LoRA  N={N}  seed=0")
    print(f"{'='*60}")

    lora_model, train_time, out_dir = train_lora_for_N(N, seed=0)

    metrics, _ = eval_ifbench(lora_model, tokenizer, eval_examples)

    r = {
        "method": "lora",
        "model": BASE_MODEL,
        "N": N,
        "seed": 0,
        **metrics,
        "train_time_sec": train_time,
        "adapter_dir": out_dir,
    }
    lora_results.append(r)
    print(f"  accuracy={r['accuracy']:.3f}  train_time={train_time:.1f}s  vram={r['peak_vram_mb']:.0f}MB")

    # Free VRAM between runs
    del lora_model
    torch.cuda.empty_cache()

## 10. Save Results

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("results/ifbench_lora")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(lora_results).sort_values(["seed", "N"])

display(df[[
    "method", "model", "seed", "N",
    "accuracy",
    "gen_tokens_per_sec", "avg_gen_time_sec", "total_gen_time_sec",
    "avg_prompt_tokens",
    "peak_vram_mb",
    "train_time_sec",
]])

out_path = RESULTS_DIR / "lora_seed0_N0_16_32_64_128_256.json"
with out_path.open("w") as f:
    json.dump(lora_results, f, indent=2)
print("Saved:", out_path)

## 11. Plots — Context-Weight Tradeoff

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_lora = pd.DataFrame([r for r in lora_results if r["method"] == "lora"]).sort_values("N")
zero_acc = zero_shot_result["accuracy"]

# --- Plot 1: Accuracy vs N (LoRA learning curve) ---
plt.figure(figsize=(7, 4))
plt.axhline(zero_acc, linestyle="--", color="gray", label=f"Zero-shot ({zero_acc:.3f})")
plt.plot(df_lora["N"], df_lora["accuracy"], marker="o", label="LoRA (seed=0)")
plt.xlabel("N (Training Examples)")
plt.ylabel("Prompt-Level Loose Accuracy")
plt.title("LoRA Accuracy vs N (IFBench, Qwen3-8B Base)")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_vs_N.png", dpi=150)
plt.show()

# --- Plot 2: Training Time vs N ---
plt.figure(figsize=(7, 4))
plt.plot(df_lora["N"], df_lora["train_time_sec"], marker="o", color="orange")
plt.xlabel("N (Training Examples)")
plt.ylabel("Training Time (seconds)")
plt.title("LoRA Training Time vs N")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "train_time_vs_N.png", dpi=150)
plt.show()

# --- Plot 3: Inference Throughput vs N (should be flat — LoRA doesn't affect inference cost) ---
plt.figure(figsize=(7, 4))
plt.plot(df_lora["N"], df_lora["gen_tokens_per_sec"], marker="o", color="green")
plt.xlabel("N (Training Examples)")
plt.ylabel("Generation Tokens / Second")
plt.title("LoRA Inference Throughput vs N (should be ~flat)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "throughput_vs_N.png", dpi=150)
plt.show()

# --- Plot 4: Peak VRAM vs N ---
plt.figure(figsize=(7, 4))
plt.plot(df_lora["N"], df_lora["peak_vram_mb"], marker="o", color="red")
plt.xlabel("N (Training Examples)")
plt.ylabel("Peak VRAM (MB)")
plt.title("LoRA Peak VRAM vs N")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "vram_vs_N.png", dpi=150)
plt.show()